# 07. Auditoría de Tags Editoriales y Propuesta de Normalización

Analizamos las etiquetas reales (`post_tag`) de los articulos de Anfibia, extraídas de la base de datos original.
Comparamos las etiquetas  frente a los **tópicos semánticos latentes** (BERTopic) para diagnosticar problemas de cobertura y solapamiento.

In [29]:
# 1. Parche de compatibilidad obligatorio
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from bertopic import BERTopic
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

# 3. Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
RAW_DATA = PROJECT_ROOT / "data" / "processed" / "posteos_anfibia.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "bertopic_hibrido_model"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_etiquetas"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de datos
df_corpus = pd.read_csv(DATA_PROCESSED)
df_tags = pd.read_csv(RAW_DATA, usecols=["ID", "post_tag"])

# Merge para traer los tags reales
df = df_corpus.merge(df_tags, on="ID", how="left")
df["doc_id"] = df.index

# 5. Carga de modelo y predicción
topic_model = BERTopic.load(MODEL_PATH)
topics, _ = topic_model.transform(df["document_text_final"].fillna("").tolist())
df["topic"] = topics

topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()
df["topic_name"] = df["topic"].map(
    lambda t: ", ".join(topic_info.get(t, "Outlier").split("_")[1:4]) if t != -1 else "Ruido(-1)"
)

print(f"Total de notas cruzadas: {len(df):,}")

Total de notas cruzadas: 3,556


## 1. Cobertura Histórica de Tags

In [30]:
total = len(df)
df["tiene_tag"] = df["post_tag"].notna() & (df["post_tag"].str.strip() != "")

con_tag = df["tiene_tag"].sum()
sin_tag = total - con_tag

print(f"Notas con etiquetas: {con_tag:,} ({(con_tag/total)*100:.1f}%)")
print(f"Notas SIN etiquetas: {sin_tag:,} ({(sin_tag/total)*100:.1f}%)")

fig = go.Figure(data=[go.Pie(labels=["Con Tags", "Sin Tags"], values=[con_tag, sin_tag], hole=.4)])
fig.update_layout(title="Cobertura de Etiquetas en el Archivo", width=500, height=400)
fig.show()

Notas con etiquetas: 1,589 (44.7%)
Notas SIN etiquetas: 1,967 (55.3%)


In [31]:


# 1. Traemos el año de publicación directo del archivo original
df_fechas = pd.read_csv(RAW_DATA, usecols=["ID", "post_year"])
df_evolucion = df.merge(df_fechas, on="ID", how="left")

# 2. Agrupamos por año: Total de notas vs Notas con tag
evolucion = (
    df_evolucion.groupby("post_year")
    .agg(
        total_notas=("doc_id", "count"),
        notas_con_tag=("tiene_tag", "sum")
    )
    .reset_index()
)
evolucion["notas_sin_tag"] = evolucion["total_notas"] - evolucion["notas_con_tag"]
evolucion["pct_con_tag"] = (evolucion["notas_con_tag"] / evolucion["total_notas"] * 100).round(1)

# Filtramos posibles años nulos o erróneos (ej. 1970)
evolucion = evolucion[evolucion["post_year"] > 2010]

# 3. Gráfico de barras apiladas
fig_evo = go.Figure()

# Barra inferior: Notas que SÍ tienen tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_con_tag"],
    name="Con Etiquetas",
    marker_color='#2ca02c',
    text=evolucion["pct_con_tag"].astype(str) + "%",
    textposition='inside'
))

# Barra superior: Notas SIN tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_sin_tag"],
    name="Sin Etiquetas (Vacío)",
    marker_color='#d62728'
))

fig_evo.update_layout(
    barmode='stack',
    title="<b>Notas con y sin Tags por Año</b>",
    xaxis_title="Año de Publicación",
    yaxis_title="Cantidad de Artículos",
    height=500,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)
fig_evo.update_xaxes(type='category') # Para que los años se vean discretos
fig_evo.write_html(OUTPUT_DIR / "03_evolucion_etiquetado.html")
fig_evo.show()

## 2. Fragmentación y Uso Único de Etiquetas
Separamos las etiquetas múltiples (divididas por `;`) y analizamos su frecuencia.

In [32]:
# Desanidar tags separados por punto y coma
tags_list = (
    df["post_tag"]
    .dropna()
    .str.split(r";")
    .explode()
    .str.strip()
    .loc[lambda x: x != ""]
)

tag_counts = tags_list.value_counts()
tags_unicos = len(tag_counts)
tags_una_vez = (tag_counts == 1).sum()

print(f"Cantidad de tags únicos creados: {tags_unicos:,}")
print(f"Tags usados en 1 sola nota: {tags_una_vez:,} ({(tags_una_vez/tags_unicos)*100:.1f}%)")

print("\nTop 15 Tags más usados:")
display(tag_counts.head(15).to_frame("Frecuencia"))

Cantidad de tags únicos creados: 881
Tags usados en 1 sola nota: 444 (50.4%)

Top 15 Tags más usados:


,Frecuencia
post_tag,
Internacional,238
Javier Milei,162
Derechas,114
Zapping cultural,84
Derechos Humanos,77
UNSAM,67
Chile,65
Feminismos,62
Peronismo,61


## 3. Solapamiento: Tags vs Tópicos BERTopic
Vemos cómo un mismo tópico semántico fue etiquetado con múltiples tags manuales distintos.

In [33]:
# Armamos df desanidado completo para agrupar
df_desanidado = df.drop("post_tag", axis=1).join(
    df["post_tag"].dropna().str.split(r";").explode().str.strip().rename("tag")
).dropna(subset=["tag"])

df_desanidado = df_desanidado[df_desanidado["tag"] != ""]

# Agrupamos por tópico
solapamiento = (
    df_desanidado[df_desanidado["topic"] != -1]
    .groupby(["topic", "topic_name"])
    .agg(
        notas_con_tag=("doc_id", "nunique"),
        tags_unicos_usados=("tag", "nunique"),
        ejemplos_tags=("tag", lambda x: " | ".join(pd.Series.unique(x)[:5]))
    )
    .reset_index()
    .sort_values("notas_con_tag", ascending=False)
)

display(solapamiento.head(10))
solapamiento.to_csv(OUTPUT_DIR / "auditoria_tags_por_topico.csv", index=False)

,topic,topic_name,notas_con_tag,tags_unicos_usados,ejemplos_tags
0,0,"político, política, electoral",113,144,Escuela de Política y Gobierno | UNSAM | Brexi...
1,1,"feminismo, feminista, patriarcado",63,66,#NiUnaMenos | LGBTIQ+ | Orgullo | Memoria tran...
2,2,"fifa, futbolista, fútbol",50,54,Perfil político | Barras bravas | Boca Juniors...
4,4,"déficit, deuda, endeudamiento",43,46,Fondos buitre | Movimientos sociales | Violenc...
5,5,"pedagógico, enseñanza, escolar",38,51,Encuentro Anfibio | Feria del Libro | Día del ...
18,18,"chileno, chile, político",36,27,Diego Maradona | Chile | Estallido en Chile | ...
13,13,"papal, catolicismo, arzobispo",32,29,Papa Francisco | Rossana Reguillo | Catolicism...
10,10,"boliviano, bolivia, político",30,43,Catolicismo | Papa Francisco | Paraguay | Reli...
16,16,"brasil, brasilero, brasilia",30,16,Brasil | Dilma Rousseff | Internacional | Mund...
8,8,"epidemia, vacunar, virus",23,24,Rosariocronica | Rosario | Covid-19 | Escuela ...


Cruzemos las Notas vs la Cantidad de Tags Únicos. Y ademas calculemos un "Índice de Fragmentación" (cuántos tags se crean por cada nota) que sirva como una tercera dimensión (color): 

si la burbuja es roja, significa que los editores inventaron casi un tag nuevo por cada artículo de ese tema.

In [34]:
import plotly.express as px

MOSTRAR_POLITICA = False

# 1. Calculamos el ratio de fragmentación sobre solapamiento
solapamiento["ratio_fragmentacion"] = (solapamiento["tags_unicos_usados"] / solapamiento["notas_con_tag"]).round(2)

# 2. Filtramos tópicos con volumen suficiente y aplicamos la flag para el Tópico 0
df_ranking = solapamiento[solapamiento["notas_con_tag"] >= 15].copy()

if not MOSTRAR_POLITICA:
    df_ranking = df_ranking[df_ranking["topic"] != 0]

# 3. Ordenamos por ratio de fragmentación (ascendente para que el peor quede arriba en barras horizontales)
df_ranking = df_ranking.sort_values("ratio_fragmentacion", ascending=True).tail(15)

# 4. Bar Chart Horizontal
fig_ranking = px.bar(
    df_ranking,
    x="ratio_fragmentacion",
    y="topic_name",
    orientation="h",
    color="ratio_fragmentacion",
    color_continuous_scale="Reds",
    text="tags_unicos_usados",
    hover_data={
        "topic": True,
        "notas_con_tag": True,
        "tags_unicos_usados": True,
        "ratio_fragmentacion": False,
        "ejemplos_tags": True,
    },
    title="<b>Proporción de etiquetas por artículo</b><br><sup>El número en la barra indica la cantidad total de tags únicos inventados para ese tema.</sup>",
    labels={
        "ratio_fragmentacion": "Índice de Caos (Tags únicos / Notas)",
        "topic_name": "Palabras del Tópico",
    },
    height=650,
    template="plotly_white",
)

# Ajustes de visualización y margen
fig_ranking.update_traces(textposition="outside")
fig_ranking.update_layout(
    xaxis_range=[0, df_ranking["ratio_fragmentacion"].max() * 1.15],
    coloraxis_showscale=False,
)

# 5. Exportar y mostrar
fig_ranking.write_html(OUTPUT_DIR / "05_ranking_caos_etiquetas.html")
fig_ranking.show()

## 4. Ideas de recomendacíon


### 4.1 Proponemos palabras de los topicos para rellenar el vacio del 56% de articulos sin etiquetas.

In [35]:
# Usamos directamente las 3 palabras clave más representativas que calculó BERTopic 
# (esto ya está en la columna 'topic_name' que creamos)
df["etiqueta_propuesta"] = df["topic_name"]

# Para los outliers (ruido estadístico), los dejamos explícitamente sin clasificar
df.loc[df["topic"] == -1, "etiqueta_propuesta"] = np.nan

# Muestra comparativa: Tag manual de la redacción vs Tag algorítmico
display(df[df["tiene_tag"]][["title_text", "post_tag", "etiqueta_propuesta"]].head(15))

# Exportar el dataset enriquecido
export_cols = ["doc_id", "ID", "title_text", "post_tag", "topic", "etiqueta_propuesta"]
df[export_cols].to_csv(OUTPUT_DIR / "corpus_con_tags_normalizados.csv", index=False)
print("Archivo 'corpus_con_tags_normalizados.csv' exportado.")

,title_text,post_tag,etiqueta_propuesta
1,El rico que se cree Steve Jobs,Perfil político,"fifa, futbolista, fútbol"
17,Fluidos trans: arte y performance queer,Memoria trans,NaN
28,¿Qué tan lejos llegan en 15 segundos?,Internacional; Israel; Palestina,"israel, netanyahu, israelí"
33,El sol de tu bravura,Internacional; Venezuela,NaN
40,La 12: historia de una traición,Barras bravas; Boca Juniors; Fútbol,"fifa, futbolista, fútbol"
45,Policías en acción,Violencia policial,NaN
64,Bolivia: Los linchados de El Alto,BA,NaN
83,Enfrentar la muerte,24M: Día de la memoria por la verdad y la just...,"asesino, testimonio, testigo"
90,La narcomáquina ya no necesita Chapos,Rossana Reguillo,NaN
92,"Crimea: allí donde haya rusos, hay Rusia",Internacional; Rusia,"soviética, soviético, vladimir"


Archivo 'corpus_con_tags_normalizados.csv' exportado.


### 4.2 Mapeo Estático Precalculado (Lookup Table)

- Etiqueta canónica precalculada: Creamos un diccionario clave-valor topic_id -> "Nombre Editorial Limpio".
- Cálculo unico: Se procesa en el pipeline de datos durante el entrenamiento/clasificación.
- Consumo online ultra rápido: Cuando el backend sirve la nota en la web, solo lee el ID del tópico asignado y devuelve el string correspondiente directo del JSON/DB.

In [36]:
# 1. Encontramos el tag manual histórico más repetido por cada tópico (Moda)
tag_canonico = (
    df_desanidado[df_desanidado["topic"] != -1]
    .groupby("topic")["tag"]
    .agg(lambda x: x.mode().iloc[0] if not x.empty else "General")
    .to_dict()
)

# 2. Asignamos la etiqueta editorial limpia al DataFrame
df["etiqueta_editorial"] = df["topic"].map(tag_canonico).fillna("Archivo")

# 3. Exportamos el diccionario estático para producción (JSON de consulta instantánea)
import json
with open(OUTPUT_DIR / "mapeo_topicos_tags.json", "w", encoding="utf-8") as f:
    json.dump(tag_canonico, f, ensure_ascii=False, indent=2)

display(df[["title_text", "topic", "etiqueta_propuesta"]].head(10))

,title_text,topic,etiqueta_propuesta
0,La China invisible,-1,NaN
1,El rico que se cree Steve Jobs,2,"fifa, futbolista, fútbol"
2,La vaca sagrada,-1,NaN
3,Los arruinados del Paraná,-1,NaN
4,De Chile a Tacna: una historia de carniceros y...,-1,NaN
5,Bolivia en llamas,10,"boliviano, bolivia, político"
6,Los líderes de la primavera mexicana (I),-1,NaN
7,Nunca te metas con las mariscadoras de Vigo,33,"submarino, marino, naufragio"
8,"Lucía, la remadora olímpica de las manos impos...",2,"fifa, futbolista, fútbol"
9,El fisicoculturista que no puede mirarse en el...,2,"fifa, futbolista, fútbol"
